# 01b — Solar Analytics Fleet EDA

Characterises the fleet: site/circuit counts, geographic spread, export limits, key-table schemas, and single-day diagnostic plots. Orchestrator — plotting lives in `lib/explore_plots.py`.

Split out of the old `01_data_exploration.ipynb` (the data-lake tooling moved to `01a`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('../shared').resolve()))
sys.path.insert(0, str(pathlib.Path('lib').resolve()))

from aws_config import aq, dread, s3_ls
from ciccada_config import SA, SAI, AS4777, TABLES
import explore_plots as ep
import pandas as pd
import numpy as np

## Sites and circuits

In [ ]:
# -----------------------------------------------------------------------------
# Assumptions:
# sites  = physical locations (one row per address)
# circuits = monitoring points within a site (one row per phase/device)
# is_pv = True means it's a solar PV circuit (not load, battery, etc.)
# -----------------------------------------------------------------------------

site_count = aq("SELECT count(*) AS n_sites FROM sites", database=SAI)
print("Total sites:", site_count["n_sites"].iloc[0])

In [ ]:
circuit_counts = aq("""
    SELECT
        is_pv,
        count(*)          AS n_circuits,
        count(DISTINCT site_id) AS n_sites
    FROM circuits
    GROUP BY is_pv
    ORDER BY is_pv DESC
""", 
database=SAI)
circuit_counts

# is_pv=True rows are what the telemetry analysis is built on.
# The values below double count sites because many sites have both PV and non-PV circuits.

In [ ]:
# How many circuits per site?
circuits_per_site = aq("""
    SELECT
        circuit_count,
        count(*) AS n_sites
    FROM (
        SELECT site_id, count(*) AS circuit_count
        FROM circuits
        WHERE is_pv = True
        GROUP BY site_id
    )
    GROUP BY circuit_count
    ORDER BY circuit_count
""", database=SAI)
circuits_per_site

## Geographic spread

In [ ]:
# -----------------------------------------------------------------------------
# Geographic spread — use meta_up23c, not sites
# -----------------------------------------------------------------------------

states = aq("""
    SELECT state, count(*) AS n_sites
    FROM meta_up23c
    GROUP BY state
    ORDER BY n_sites DESC
""", database=SAI)
states

In [ ]:
# -----------------------------------------------------------------------------
# Use partition_lookup (tiny table) rather than querying ts directly.
# Reading min/max timestamps from a billions-row table is expensive;
# the lookup table gives you the answer for free.
# -----------------------------------------------------------------------------

partitions = aq("SELECT * FROM partition_lookup ORDER BY year, month", database=SA)
partitions
# Each row = one (year, month) partition that exists in the ts table.
# The first and last rows tell you the data window.

## Export limits (flex_export_detected)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# flex_export_detected diagnostic checks
# Paste these cells into your notebook after the prevalence check
# ═══════════════════════════════════════════════════════════════════════════════

# %% ── Check 1: Where are the flagged sites? ─────────────────────────────────
# Break down by state and DNSP to see if flex export is concentrated
# (e.g. SA Power Networks runs a major flexible export program)

flex_by_dnsp = aq("""
    SELECT
        flex_export_detected,
        state,
        dnsp_name,
        count(DISTINCT site_id)  AS n_sites,
        round(avg(ac_capacity_kw), 1) AS avg_ac_kw,
        round(avg(export_limit_kw), 1) AS avg_export_limit_kw
    FROM meta_up23c
    WHERE is_pv = True
    GROUP BY flex_export_detected, state, dnsp_name
    ORDER BY flex_export_detected DESC, n_sites DESC
""", database='solar_analytics_iceberg')

print("Flex-export sites by state and DNSP:")
print(flex_by_dnsp.to_string(index=False))

# %% ── Check 2: Do flagged sites have explicit export limits? ─────────────────
# If export_limit_kw is set AND is less than ac_capacity_kw, that's a strong
# signal the site really is DOE-constrained.

flex_export_limits = aq("""
    SELECT
        flex_export_detected,
        count(DISTINCT site_id) AS n_sites,
        sum(CASE WHEN export_limit_kw IS NOT NULL THEN 1 ELSE 0 END) AS has_export_limit,
        sum(CASE WHEN export_limit_kw IS NOT NULL
                  AND export_limit_kw < ac_capacity_kw THEN 1 ELSE 0 END)
            AS export_limit_below_nameplate
    FROM (
        SELECT DISTINCT site_id, ac_capacity_kw, export_limit_kw, flex_export_detected
        FROM meta_up23c
        WHERE is_pv = True
    )
    GROUP BY flex_export_detected
""", database='solar_analytics_iceberg')

print("\nExport limit breakdown:")
print(flex_export_limits.to_string(index=False))

## Schema of the key tables

In [ ]:
# Open up schemas here:
schemas['meta_up23c']

## Fleet characteristics

In [ ]:
# -----------------------------------------------------------------------------
# Site summary four separate queries, one per count
# (Athena can't mix two databases in a single query)
# -----------------------------------------------------------------------------

n_sites         = aq("SELECT count(DISTINCT site_id) AS n FROM sites",       database=SAI)["n"].iloc[0]
n_meta_up23c    = aq("SELECT count(DISTINCT site_id) AS n FROM meta_up23c",  database=SAI)["n"].iloc[0]
n_pv_circuits   = aq("SELECT count(DISTINCT site_id) AS n FROM circuits WHERE is_pv = True", database=SAI)["n"].iloc[0]
#n_single_inv    = aq("SELECT count(*) AS n FROM meta_single_inverters",       database=SA) ["n"].iloc[0]

site_summary = pd.DataFrame([{
    "total_in_sites_table":       n_sites,
    "sites_in_meta_up23c":        n_meta_up23c,
    "sites_with_pv_circuit":      n_pv_circuits
#    "meta_single_inverters_rows": n_single_inv,
}])
site_summary

## Single-day diagnostic plots

Pick a site + date, pull one day, convert to AEST, and plot. Two views:
- `plot_operational` — Volt-Watt + Volt-VAr response
- `plot_protective` — sustained-operation + anti-islanding (over/under-voltage)

In [ ]:
# 1. choose a site + day
SITE_ID   = 276149647          # example
ZOOM_DATE = '2024-01-15'
AC_CAP    = 15.0               # nameplate kW for this site

# 2. pull one day of telemetry (site-level, max voltage across circuits)
df_day = aq(f'''
    WITH sm AS (
        SELECT DISTINCT circuit_id, circuit_polarity
        FROM meta_up23c WHERE is_pv = True AND site_id = {SITE_ID}
    )
    SELECT t.t_stamp,
           max(t.voltage)                              AS voltage,
           sum(t.power * sm.circuit_polarity)/1000     AS P_kW,
           sum(t.energy_reactive*sm.circuit_polarity)/1000*12 AS Q_kvar
    FROM ts t JOIN sm ON t.circuit_id = sm.circuit_id
    WHERE t.year = {int(ZOOM_DATE[:4])} AND t.month = {int(ZOOM_DATE[5:7])}
      AND t.is_pv = True
      AND date(t.t_stamp + interval '10' hour) = date '{ZOOM_DATE}'
    GROUP BY t.t_stamp ORDER BY t.t_stamp
''', database=SAI)

# 3. AEST column the plotters expect
df_day['t_stamp_aest'] = ep.to_aest(df_day['t_stamp'])
len(df_day)

In [ ]:
ep.plot_operational(df_day, SITE_ID, AC_CAP, ZOOM_DATE, AS4777)

In [ ]:
# protective thresholds default to AS/NZS 4777.2 (258/265/275/180/70 V)
ep.plot_protective(df_day, SITE_ID, AC_CAP, ZOOM_DATE)